In [0]:
%run ./transform_data

In [0]:
df_date = missing_asset_measure_with_tg_code.withColumn(
    "calculation_date",
    to_date(col("calculation_interval_after"))
)

In [0]:
result = df_date.select(
    "batch_id", "production_line", "prd_workshop", "prd_cell", "measure_calculation_datetime", "measure_name", "measure_value", "tagname","calculation_interval_after", "localizations_used","mean_occurrences", "map_hour_occurrences", "is_missing_occurrences", "target_localization", "localizations_used", "is_late","calculation_date"
)

result = result.dropDuplicates()

In [0]:
fact_ascendance_localization = result \
    .withColumn("measure_value", F.col("measure_value").cast("float")) \
    .withColumn("mean_occurrences", F.col("mean_occurrences").cast("float"))

Ingestion des données dans la table cible

In [0]:
current_process= "fact_ascendance_localization"

In [0]:
target_fact_ascendance_localization = current_catalog +"."+current_schema+"."+current_process
print(target_fact_ascendance_localization)

In [0]:
all_columns =  fact_ascendance_localization.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'batch_id'
    ,'tagname'
    ,'measure_name'
    ,'target_localization']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    fact_ascendance_localization, 
    target_fact_ascendance_localization, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )